<a href="https://www.kaggle.com/code/mugishapacifique/timeseries-blending?scriptVersionId=322208764" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Time Series Blending
## Walk-forward OOF Stacking · Dynamic 1/Error Weighting

This notebook walks through two ensemble blending methods for time series — **without data leakage**.

| Cell | What it covers |
|------|---------------|
| 1 | Synthetic data (trend + seasonality + regime shift) |
| 2 | Three base models |
| 3 | Strict walk-forward OOF generation |
| 4 | Meta-model trained on OOF predictions |
| 5 | Dynamic 1/error weighting at inference |
| 6 | OOF stacking at inference |
| 7 | MAE comparison |
| 8 | Visualisations |

---


## Cell 1 — Imports & Synthetic Data

We build a series with three components so each model has different strengths:
- **Trend** — benefits linear regression
- **Seasonality** — benefits exponential smoothing & moving average  
- **Regime shift at t=130** — tests adaptability of dynamic weighting


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import Ridge

np.random.seed(42)

n = 200
t = np.arange(n)
y = (
    0.05 * t                              # slow upward trend
    + 3.0 * np.sin(2 * np.pi * t / 20)   # repeating cycle  (period = 20 steps)
    + np.where(t >= 130, 5.0, 0.0)       # level jump at t=130  ← regime change
    + np.random.normal(0, 0.8, n)        # noise
)
dates = pd.date_range("2020-01-01", periods=n, freq="W")
series = pd.Series(y, index=dates, name="value")

# Quick look
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(dates, y, color="#2C2C2A", lw=1.2, alpha=0.8)
ax.axvline(dates[130], color="#D85A30", lw=1.0, linestyle="--", label="regime shift (t=130)")
ax.set_title("Synthetic time series", fontweight="bold")
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(f"Length: {n}  |  mean: {y.mean():.2f}  |  std: {y.std():.2f}")


---
## Cell 2 — Three Base Models

Each model is a **pure function**:  `f(history: np.array) → float`  
It receives everything up to time *t* and returns one prediction for *t+1*.

| Model | Strength | Weakness |
|-------|----------|----------|
| Moving average | Stable, noise-resistant | Lags behind trend & regime shifts |
| Exponential smoothing | Adapts to level changes | Struggles with strong seasonality |
| Linear regression on lags | Captures trend & pattern | Overfits on short series |


In [ ]:
def model_moving_avg(history, window=6):
    """Simple average of the last `window` observations."""
    return float(np.mean(history[-window:]))


def model_exp_smooth(history, alpha=0.35):
    """
    Exponential smoothing — recent points carry more weight.
    alpha close to 1 → very reactive; close to 0 → very smooth.
    """
    s = history[0]
    for x in history[1:]:
        s = alpha * x + (1 - alpha) * s
    return float(s)


def model_linreg(history, lags=10):
    """
    Fit ordinary least squares on lag features at every step,
    then predict the next value.
    Falls back to last value if history is too short.
    """
    if len(history) < lags + 2:
        return float(history[-1])
    X    = np.array([history[i - lags:i] for i in range(lags, len(history))])
    y_tr = history[lags:]
    coef = np.linalg.lstsq(
        np.column_stack([X, np.ones(len(X))]), y_tr, rcond=None
    )[0]
    feat = np.append(history[-lags:], 1.0)
    return float(feat @ coef)


MODELS = {
    "MovAvg":    model_moving_avg,
    "ExpSmooth": model_exp_smooth,
    "LinReg":    model_linreg,
}
print("Base models defined:", list(MODELS.keys()))


---
## Cell 3 — Strict Walk-Forward OOF (No Leakage)

### Why naive k-fold leaks on time series
Standard k-fold randomly picks a fold to hold out and trains on the rest.  
For time series this means a model can train on *future* data to predict the *past* — that's leakage.

### The fix: strict walk-forward
At every step **t**, each model trains on `y[0 : t]` (past only) and predicts `y[t]`.  
The window grows forward one step at a time — the future is never visible.

```
t=30 : train on y[0..29]  → predict y[30]   ← OOF prediction #1
t=31 : train on y[0..30]  → predict y[31]   ← OOF prediction #2
...
t=199: train on y[0..198] → predict y[199]  ← OOF prediction #170
```

After the loop every position `t ≥ MIN_TRAIN` has an honest held-out prediction  
from a model that **never saw `y[t]`** during training.


In [ ]:
MIN_TRAIN = 30   # warm-up steps before we start generating OOF predictions

oof_preds = {name: np.full(n, np.nan) for name in MODELS}

for t_idx in range(MIN_TRAIN, n):
    history = y[:t_idx]                   # ← strictly causal, past only
    for name, fn in MODELS.items():
        oof_preds[name][t_idx] = fn(history)

# ── per-model OOF MAE ────────────────────────────────────────────
print("OOF walk-forward MAE per base model:")
for name in MODELS:
    valid = ~np.isnan(oof_preds[name])
    mae   = np.mean(np.abs(oof_preds[name][valid] - y[valid]))
    print(f"  {name:12s}  MAE = {mae:.4f}")


---
## Cell 4 — Train Meta-Model on OOF Predictions

Now we stack the OOF columns as features and train a Ridge regression  
to learn the optimal combination.

```
X_meta  →  [ oof_MovAvg | oof_ExpSmooth | oof_LinReg ]   (shape: valid_steps × 3)
y_meta  →  actual y                                        (shape: valid_steps,)
```

Because every row in `X_meta` was predicted *without* seeing `y_meta[row]`,  
there is **zero leakage** — the meta-model sees only honest test predictions.

### Why retrain base models at inference?
The meta-model weights are **frozen** after this cell.  
What changes at inference is that each base model's `history` grows to include  
all available data — so its raw predictions improve, even though the  
blend formula never changes.  Think of it as: *same recipe, fresher ingredients.*


In [ ]:
valid_mask = ~np.isnan(oof_preds["MovAvg"])   # identical for all models
X_meta = np.column_stack([oof_preds[name][valid_mask] for name in MODELS])
y_meta = y[valid_mask]

meta = Ridge(alpha=0.5, fit_intercept=False)
meta.fit(X_meta, y_meta)

weights_meta = dict(zip(MODELS.keys(), meta.coef_))
print("Frozen meta-model weights (used at inference, never updated again):")
for name, w in weights_meta.items():
    print(f"  {name:12s}  weight = {w:+.4f}")

print()
print("Note: weights can be negative — Ridge allows one model to 'correct'")
print("another by subtracting its bias, which is often statistically valid.")


---
## Cell 5 — Dynamic 1/Error Weighting

Instead of a fixed meta-model, here we **recompute blend weights at every step**  
using each model's recent error history.

### Algorithm (per step t)

```
1. window  = oof_preds[:, t-WINDOW : t]   ← honest OOF predictions for last WINDOW steps
2. MAE_i   = mean |oof_preds[i] - actual|  ← per-model rolling error
3. raw_w_i = 1 / MAE_i                     ← worse model → smaller weight
4. w_i     = raw_w_i / Σ raw_w_j           ← normalise so weights sum to 1
5. ŷ_t     = Σ w_i · model_i(history)      ← weighted blend
```

### Key properties
- **No retraining needed** — weights update with a simple formula  
- **Adaptive** — if a model degrades after a regime shift, its weight shrinks within `WINDOW` steps  
- **Uses OOF errors** to measure performance honestly (no leakage in the rolling window either)


In [ ]:
WINDOW = 20   # how many past steps to measure rolling error over

dynamic_blends  = np.full(n, np.nan)
weight_history  = []   # store per-step weight dicts for plotting

for t_idx in range(MIN_TRAIN, n):
    history   = y[:t_idx]
    win_start = max(MIN_TRAIN, t_idx - WINDOW)
    win_len   = t_idx - win_start

    # ── Step A: current predictions ──────────────────────────────
    step_preds = {name: fn(history) for name, fn in MODELS.items()}

    # ── Step B: rolling MAE using pre-computed OOF errors ────────
    if win_len < 3:
        # too little history → equal weights
        w = {name: 1.0 / len(MODELS) for name in MODELS}
    else:
        win_actual = y[win_start:t_idx]
        raw_w = {}
        for name in MODELS:
            win_preds = oof_preds[name][win_start:t_idx]
            valid     = ~np.isnan(win_preds)
            if valid.sum() < 2:
                raw_w[name] = 1.0          # fallback: neutral weight
            else:
                mae           = np.mean(np.abs(win_preds[valid] - win_actual[valid]))
                raw_w[name]   = 1.0 / (mae + 1e-9)   # epsilon avoids division by zero

        # ── Step C: normalise ─────────────────────────────────────
        total = sum(raw_w.values())
        w     = {name: raw_w[name] / total for name in MODELS}

    # ── Step D: weighted blend ────────────────────────────────────
    dynamic_blends[t_idx] = sum(w[name] * step_preds[name] for name in MODELS)
    weight_history.append(w)

print(f"Dynamic blending complete.  Non-NaN predictions: {(~np.isnan(dynamic_blends)).sum()}")
print()
print("Sample weights at t=50 (pre-regime):", {k: f"{v:.3f}" for k,v in weight_history[20].items()})
print("Sample weights at t=150 (post-regime):", {k: f"{v:.3f}" for k,v in weight_history[120].items()})


---
## Cell 6 — OOF Stacking at Inference

The meta-model is frozen. At every new step we:
1. Ask each base model to predict using all past data  
2. Pass those three predictions through the frozen Ridge meta-model  
3. Output the blend

This is identical to what the meta-model saw during training, except the base models  
now have more history — which is the whole point of retraining on full data.


In [ ]:
stack_blends = np.full(n, np.nan)

for t_idx in range(MIN_TRAIN, n):
    history    = y[:t_idx]
    step_preds = np.array([fn(history) for fn in MODELS.values()])
    stack_blends[t_idx] = meta.predict([step_preds])[0]

print(f"OOF stacking inference complete.  Non-NaN predictions: {(~np.isnan(stack_blends)).sum()}")


---
## Cell 7 — MAE Comparison

We compare all methods on the same evaluation window `[MIN_TRAIN, n)`.


In [ ]:
eval_idx = np.arange(MIN_TRAIN, n)

def mae(preds):
    return np.mean(np.abs(preds[eval_idx] - y[eval_idx]))

results = {
    **{name: mae(oof_preds[name]) for name in MODELS},
    "OOF stack blend": mae(stack_blends),
    "Dynamic 1/err":   mae(dynamic_blends),
}

print("=" * 42)
print(f"{'Method':<22}  {'MAE':>8}")
print("-" * 42)
best = min(results.values())
for method, m in sorted(results.items(), key=lambda x: x[1]):
    flag = "  ◀ best" if m == best else ""
    print(f"  {method:<20}  {m:>8.4f}{flag}")
print("=" * 42)
print()
print("Interpretation:")
print("  OOF stacking often wins overall because the meta-model finds a globally")
print("  optimal combination. Dynamic weighting can close the gap — or even win —")
print("  on series with frequent regime changes, because it adapts in real time.")


---
## Cell 8 — Visualisations

Three panels:
1. **Actual vs blends** — how well each method tracks the series  
2. **Dynamic weights over time** — watch them shift around the regime change at t=130  
3. **Prediction errors** — dynamic weighting should recover faster after the jump


In [ ]:
COLORS = {
    "MovAvg":          "#888780",
    "ExpSmooth":       "#885DA8",
    "LinReg":          "#D85A30",
    "OOF stack blend": "#185FA5",
    "Dynamic 1/err":   "#1D9E75",
}
REGIME = dates[130]
ev     = slice(MIN_TRAIN, n)

fig = plt.figure(figsize=(14, 11))
fig.patch.set_facecolor("#F9F9F7")
gs  = gridspec.GridSpec(3, 1, hspace=0.5)

# ── Panel 1: Actual vs blend methods ─────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor("#F9F9F7")
ax1.plot(dates, y, color="black", lw=1.1, alpha=0.45, label="Actual")
ax1.plot(dates[ev], stack_blends[ev],
         color=COLORS["OOF stack blend"], lw=1.6, label="OOF stack blend")
ax1.plot(dates[ev], dynamic_blends[ev],
         color=COLORS["Dynamic 1/err"], lw=1.6, ls="--", label="Dynamic 1/err")
ax1.axvline(REGIME, color="#D85A30", lw=1.0, ls=":", alpha=0.8)
ax1.text(REGIME, ax1.get_ylim()[1] * 0.97, " regime shift",
         fontsize=8, color="#D85A30", va="top")
ax1.set_title("Blend methods vs actual", fontsize=11, fontweight="bold")
ax1.legend(fontsize=9, framealpha=0.4); ax1.grid(alpha=0.2)

# ── Panel 2: Dynamic weights over time ───────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.set_facecolor("#F9F9F7")
for name in MODELS:
    w_vals = [dw[name] for dw in weight_history]
    ax2.plot(dates[ev], w_vals, label=name, color=COLORS[name], lw=1.3)
ax2.axvline(REGIME, color="#D85A30", lw=1.0, ls=":", alpha=0.8)
ax2.set_ylim(-0.05, 1.05)
ax2.set_title("Dynamic weights over time  — watch them shift at the regime change",
              fontsize=11, fontweight="bold")
ax2.legend(fontsize=9, framealpha=0.4); ax2.grid(alpha=0.2)
ax2.set_ylabel("weight")

# ── Panel 3: Errors ───────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
ax3.set_facecolor("#F9F9F7")
ax3.plot(dates[ev], stack_blends[ev]  - y[MIN_TRAIN:],
         color=COLORS["OOF stack blend"], lw=1.0, alpha=0.85, label="OOF stack error")
ax3.plot(dates[ev], dynamic_blends[ev] - y[MIN_TRAIN:],
         color=COLORS["Dynamic 1/err"], lw=1.0, ls="--", alpha=0.85, label="Dynamic error")
ax3.axhline(0, color="black", lw=0.5)
ax3.axvline(REGIME, color="#D85A30", lw=1.0, ls=":", alpha=0.8)
ax3.set_title("Prediction errors — dynamic should recover faster post-shift",
              fontsize=11, fontweight="bold")
ax3.legend(fontsize=9, framealpha=0.4); ax3.grid(alpha=0.2)
ax3.set_ylabel("error")

plt.savefig("ts_blending_results.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Figure saved → ts_blending_results.png")


---
## Summary

| Concept | Key point |
|---------|-----------|
| **Walk-forward OOF** | Each step t trains strictly on `y[0:t]` — no future fold ever contaminates training |
| **Meta-model** | Learns optimal blend weights from honest OOF predictions; frozen after training |
| **Retraining base models** | Meta weights are frozen; base models benefit from seeing all data at inference |
| **Dynamic 1/error** | Recomputes weights every step from rolling MAE; no meta-model needed |
| **When to prefer stacking** | Stable series where global weights generalise well |
| **When to prefer dynamic** | Series with frequent regime changes needing fast adaptation |
| **Combining both** | Use OOF stacking for initial weights, allow dynamic adjustment within a bounded range |
